DATA CLEANING

PART 2: CURATED LAYER ------------------------------------------------------------------------------------------------------------------------

Begin Our PySpark Session.

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/17 23:42:59 WARN Utils: Your hostname, DESKTOP-9UAA9AK, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/17 23:42:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 23:43:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/17 23:43:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Lets define the null check function once more.

In [2]:
from functools import reduce

def null_check(df):
    has_null_condition = reduce(
        lambda acc, c: acc | F.col(c).isNull(),
        df.columns,
        F.lit(False)
    )

    # Count total rows, rows with atleast one null and count how many 
    # nulls occur per data field.
    summary = df.agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.sum(has_null_condition.cast("int")).alias(
            "rows_with_at_least_one_null"),
        *[F.sum(F.col(c).isNull().cast("int")).alias(
            f"null_{c}") for c in df.columns]
    ).collect()[0]
    
    total = summary["total_rows"]
    rows_with_at_least_one_null = summary["rows_with_at_least_one_null"]
    
    print(f"Total rows: {total:,}")
    print(f"Rows with at least one null value: \
          {rows_with_at_least_one_null:,}")
    print(f"Percentage of rows with atleast one NULL: {round(
        100 * rows_with_at_least_one_null / total, 2)}%")
    print()
    for c in df.columns:
        print(f"{c:25s} {summary[f'null_{c}']:>12,}")

<br>
<br>
<br>

### Dealing with `structured_features`

Lets clean up our `structued_features` descriptions so they are all comma seperated properly and consistent with other rows.

**Reload raw data.** Reads the raw parquet file (`vic_rentals_raw.parquet`) into a Spark DataFrame `df_raw` and prints its row/column count. This re-loads a fresh copy to work on for this stage of cleaning.

In [3]:
raw_path = "../../rea_data/raw/vic_rentals_raw.parquet"

df_raw = spark.read.parquet(raw_path)

print(f"Shape: {df_raw.count():,} rows, {len(df_raw.columns)} columns")

Shape: 12,717 rows, 31 columns


**Inspect a known bad row.** Filters to a specific listing (`17747859`) that was manually spotted as having a formatting problem in `structured_features`, and prints it vertically so every field is readable. This is a "before" snapshot for the fix in the next cell.

In [4]:
# This error was found upon visual inspection of the csv
MALFORMED_LISTING_ID = "17747859"

row = df_raw.filter(F.col("listing_id") == MALFORMED_LISTING_ID)
row.show(truncate=False, vertical=True)

-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 listing_id          | 17747859                                                                                                                                                                                                                                                                                                                                  
 suburb              | DOCKLANDS                                                                                                                                                                                                                                                                    

**Fix bullet-point line breaks.** Uses a regex (`[\r\n]+-\s*`) to find bullet-style line breaks (`\n- `) inside the `structured_features` text and replaces them with a plain comma. This normalizes messy multi-line bullet lists into a single comma-separated string.

In [5]:
df_raw = df_raw.withColumn(
    "structured_features",
    F.regexp_replace(
        F.col("structured_features"),
        r"[\r\n]+-\s*",   # matches \r\n or \n followed by "- " 
        ", "              # replace with a plain comma separator
    )
)

**Re-check the same row.** Re-filters the same listing (`17747859`) to visually confirm the regex fix worked — the "after" snapshot.

In [6]:
row = df_raw.filter(F.col("listing_id") == MALFORMED_LISTING_ID)
row.show(truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 listing_id          | 17747859                                                                                                                                                                                                                                                                                                                  
 suburb              | DOCKLANDS                                                                                                                                                                                                                                                                                                    

We used a specific case ("listing_id=17747859") to illustrate our point, but must check that it has been fixed for every such row.

**Verify the fix generalizes.** Searches the whole dataset for any remaining rows still matching the old bullet pattern (`\n- `), and counts them, to confirm the fix wasn't a one-off patch that only worked on the example row.

In [7]:
remaining_bullets = df_raw.filter(
    F.col("structured_features").rlike(r"[\r\n]-\s")
)
remaining_bullets.count()

0

**Parse features into an array.** Three steps combined: (1) strips section headers like "BUILDING AMENITIES" out of `structured_features`; (2) splits the cleaned string on commas into an array, trimming whitespace and lowercasing each element, then drops empty strings; (3) explodes that array (one row per listing-feature pair) and counts how many times each distinct feature string appears — giving a frequency table of every unique feature value in the dataset.

In [8]:
from pyspark.ml.feature import CountVectorizer

# --- 1. Strip section headers (e.g. "BUILDING AMENITIES") before splitting ---
header_pattern = r"(BUILDING AMENITIES|INDOOR FEATURES|OUTDOOR FEATURES\
    |CLIMATE CONTROL)\s*,?\s*"
df_raw = df_raw.withColumn(
    "structured_features_clean",
    F.regexp_replace(F.col("structured_features"), header_pattern, "")
)

# --- 2. Split into array, trim whitespace, lowercase each element ---
df_raw = df_raw.withColumn(
    "features_array",
    F.expr("""
        transform(
            split(trim(structured_features_clean), ','),
            x -> lower(trim(x))
        )
    """)
).withColumn(
    "features_array",
    # drop empty strings from trailing commas
    F.expr("filter(features_array, x -> x != '')")  
)

# --- Explode features_array into one row per (listing, feature), 
# then count distinct values ---
feature_counts = (
    df_raw
    .select(F.explode("features_array").alias("feature"))
    .groupBy("feature")
    .count()
    .orderBy(F.desc("count"))
)

total_unique = feature_counts.count()
print(f"Total unique feature strings: {total_unique}")

feature_counts.show(total_unique, vertical = True, truncate=True)

row = df_raw.filter(F.col("listing_id") == "17747859")
row.show(truncate=True, vertical=True)

Total unique feature strings: 532
-RECORD 0-----------------------
 feature | built in wardrobes   
 count   | 6690                 
-RECORD 1-----------------------
 feature | heating              
 count   | 5710                 
-RECORD 2-----------------------
 feature | dishwasher           
 count   | 5456                 
-RECORD 3-----------------------
 feature | secure parking       
 count   | 4966                 
-RECORD 4-----------------------
 feature | internal laundry     
 count   | 3993                 
-RECORD 5-----------------------
 feature | air conditioning     
 count   | 3796                 
-RECORD 6-----------------------
 feature | balcony / deck       
 count   | 2509                 
-RECORD 7-----------------------
 feature | gas                  
 count   | 2421                 
-RECORD 8-----------------------
 feature | floorboards          
 count   | 1904                 
-RECORD 9-----------------------
 feature | bath                 
 count   

**Flag likely junk features.** Adds heuristic flags to the feature-count table to separate genuine short features (e.g. "balcony") from junk — long free-text phrases that leaked in from listing descriptions. Flags anything with >6 words, >45 characters, embedded sentence punctuation, or phrases like "will be"/"please"/"provided". Prints both groups separately so they can be manually reviewed before any rows are dropped.

In [9]:
# --- Junk filter: flag likely listing-description leakage 
# vs genuine short features ---

feature_counts_flagged = (
    feature_counts
    .withColumn("word_count", F.size(F.split(F.col("feature"), r"\s+")))
    .withColumn("char_length", F.length(F.col("feature")))
    .withColumn(
        "is_likely_junk",
        (F.col("word_count") > 6) |                                  
        # long free-text phrases
        (F.col("char_length") > 45) |                                 
        # long strings generally
        F.col("feature").rlike(r"\.\s|,\s.*,\s") |                    
        # embedded sentence punctuation
        F.col("feature").rlike(r"\b(will be|please|provided|included at|\
            works during|suitable for)\b")
    )
)

# --- Review what gets flagged before deleting anything ---
print("=== Flagged as likely junk (review before dropping) ===")
feature_counts_flagged.filter("is_likely_junk").orderBy(F.desc("count")).show(
    100, truncate=False)

print("=== Kept as likely genuine features ===")
feature_counts_flagged.filter("NOT is_likely_junk").orderBy(
    F.desc("count")).show(100, truncate=False)

n_junk = feature_counts_flagged.filter("is_likely_junk").count()
n_kept = feature_counts_flagged.filter("NOT is_likely_junk").count()
print(f"\nFlagged as junk: {n_junk} / {n_junk + n_kept} unique features")

=== Flagged as likely junk (review before dropping) ===
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+-----------+--------------+
|feature                                                                                                                                                                                                                                |count|word_count|char_length|is_likely_junk|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------+-----------+--------------+
|energy eff. rating: 6                                                                                        

**Extract energy rating as a number.** Finds array elements matching the pattern "energy eff. rating: X" and pulls out the numeric value X into a new `energy_eff_rating` column (as a double), then removes those raw strings from `features_array` now that the value's captured elsewhere. Ends with a sanity check: a sample of non-null ratings, and summary stats (count/min/max/mean).

In [ ]:
# --- Extract "energy eff. rating: X" into a proper numeric column ---

df_raw = df_raw.withColumn(
    "energy_eff_rating",
    F.expr("""
        aggregate(
            filter(features_array, x -> x rlike 'energy eff\\\\. rating: [0-9.]+'),
            cast(null as double),
            (acc, x) -> try_cast(regexp_extract(x, 'energy eff\\\\. rating: ([0-9.]+)', 1) as double)
        )
    """)
)

# --- Remove the raw "energy eff. rating: X" strings from features_array now that they're captured ---
df_raw = df_raw.withColumn(
    "features_array",
    F.expr("filter(features_array, x -> NOT (x rlike 'energy eff\\\\. rating: [0-9.]+'))")
)

# --- Sanity check ---
df_raw.select("listing_id", "energy_eff_rating").filter(
    F.col("energy_eff_rating").isNotNull()).show(15)
df_raw.select("energy_eff_rating").summary("count", "min", "max", "mean").show()

+----------+-----------------+
|listing_id|energy_eff_rating|
+----------+-----------------+
+----------+-----------------+

+-------+-----------------+
|summary|energy_eff_rating|
+-------+-----------------+
|  count|                0|
|    min|             NULL|
|    max|             NULL|
|   mean|             NULL|
+-------+-----------------+



**Set up fuzzy matching prep.** Converts the small feature-count table to pandas (only ~532 rows, so this is cheap). Splits features into two groups: ones containing digits or starting with "un-" (e.g. "unfurnished") are excluded from fuzzy clustering since they need different handling (numeric extraction or explicit opposite-category treatment), and everything else goes into `fuzzy_df` for clustering.

In [11]:
# Collect feature counts to pandas (only 532 rows — trivial size)
pdf_counts = feature_counts.toPandas()

# pip install rapidfuzz --break-system-packages   # if not already installed
from rapidfuzz import fuzz, process

# --- Exclude features with digits or "un-" prefixes from fuzzy clustering ---
# These need separate handling: digits -> numeric columns, un-prefixed -> 
# opposite category, not a fuzzy merge
exclude_mask = pdf_counts["feature"].str.contains(r"\d", regex=True) | \
    pdf_counts["feature"].str.match(r"^un\w+")
excluded_df = pdf_counts[exclude_mask]
fuzzy_df = pdf_counts[~exclude_mask]

print(f"Excluded from fuzzy clustering: {len(excluded_df)} (review separately)")
excluded_df.sort_values("count", ascending=False)

Excluded from fuzzy clustering: 52 (review separately)


,feature,count
35,pool/spa count: 0,253
120,energy eff. rating: 6,20
154,garage spaces: 1,10
179,energy eff. rating: 5,7
183,energy eff. rating: 7,6
184,unfurnished,6
185,car spaces: 1,6
215,garage spaces: 2,4
219,energy eff. rating: 6.5,4
227,carport spaces: 1,4


**Fuzzy-cluster similar feature strings.** Uses `rapidfuzz` to group near-duplicate feature strings (e.g. "split system aircon" vs "split-system air con") by token-sort similarity (score ≥ 87), processing most-frequent terms first so they become the canonical cluster name. Prints the candidate clusters for manual review — this is diagnostic, not yet applying any changes.

In [12]:
features = fuzzy_df["feature"].tolist()
counts = dict(zip(fuzzy_df["feature"], fuzzy_df["count"]))

clusters = {}
assigned = set()

# Process from most-frequent to least, so common terms become 
# canonical cluster labels
for feat in sorted(features, key=lambda f: -counts[f]):
    if feat in assigned:
        continue
    matches = process.extract(
        feat, features, scorer=fuzz.token_sort_ratio, limit=10, score_cutoff=87
    )
    group = [m[0] for m in matches if m[0] not in assigned]
    if len(group) > 1:
        clusters[feat] = group  # feat = canonical name (most frequent form)
        assigned.update(group)

# Show candidate clusters for manual review
for canonical, group in clusters.items():
    if len(group) > 1:
        print(f"{canonical!r}  <-  {group}")

'internal laundry'  <-  ['internal laundry', 'laundry: internal']
'balcony / deck'  <-  ['balcony / deck', 'deck / balcony']
'remote garage'  <-  ['remote garage', 'garage remote']
'ducted heating'  <-  ['ducted heating', 'gas ducted heating']
'split system heating'  <-  ['split system heating', 'split system (heating)', 'split-system heating']
'outdoor entertainment area'  <-  ['outdoor entertainment area', 'outdoor entertaining area', 'outdoor entertainment']
'courtyard'  <-  ['courtyard', 'coutyard']
'split system air con'  <-  ['split system air con', 'split system aircon', 'split system (air con)', 'split-system air con']
'split system air conditioning'  <-  ['split system air conditioning', 'split system airconditioning', 'split-system air conditioning', 'split-system air-conditioning']
'gas heating'  <-  ['gas heating', 'heating: gas']
'cooking: gas hotplates and electric oven'  <-  ['cooking: gas hotplates and electric oven', 'cooking: electric hotplates & gas oven', 'cooking: 

**Define the canonical merge mapping.** A hand-built dictionary (`merge_map`) that maps messy/duplicate feature-string variants to a single canonical spelling (e.g. all the "split system air con" variants → `"split system air conditioning"`). Includes inline notes flagging a couple of judgment calls (e.g. an assumed fuel type, a canonical name that itself contains a typo).

In [13]:
merge_map = {
    # --- Formatting/spelling variants -> canonical name ---
    "laundry: internal": "internal laundry",
    "deck / balcony": "balcony / deck",
    "garage remote": "remote garage",
    "gas ducted heating": "ducted heating",
    "split system (heating)": "split system heating",
    "split-system heating": "split system heating",
    "outdoor entertaining area": "outdoor entertainment area",
    "outdoor entertainment": "outdoor entertainment area",
    "coutyard": "courtyard",

    # split system air con group 
    # (merged together — "air con" == "air conditioning")
    "split system aircon": "split system air conditioning",
    "split system (air con)": "split system air conditioning",
    "split-system air con": "split system air conditioning",
    "split system air con": "split system air conditioning",
    "split system airconditioning": "split system air conditioning",
    "split-system air conditioning": "split system air conditioning",
    "split-system air-conditioning": "split system air conditioning",
    "split system ac": "split system air conditioning",
    "split system": "split system air conditioning",

    "heating: gas": "gas heating",
    "cooking: electric hotplates & gas oven": \
        "cooking: gas hotplates and electric oven",
    "cooking: electric hotplates & oven": \
        "cooking: gas hotplates and electric oven",
    "cooking: gas oven and hotplates": \
        "cooking: gas hotplates and electric oven",
    "reverse cycle aircon": "reverse cycle air con",
    "ducted air-conditioning": "ducted air conditioning",
    "tennis courts": "tennis court",
    "spa (inside)": "inside spa",
    "modern kitchen": "kitchen: modern",
    "basement car parking": "car parking - basement",
    "living areas": "living area",
    "hot water service": "gas hot water service",  
    # NOTE: see caveat below — fuel type unconfirmed
    "kitchen: renovated": "renovated kitchen",
    "laundry: separate": "separate laundry",

    # in-ground pool group 
    # (kept SEPARATE from above-ground — see correction below)
    "pool (inground)": "inground pool",
    "pool (in ground)": "inground pool",

    # above-ground pool group (corrected: no longer merged with "below ground")
    "pool (above ground)": "above ground pool",
    "above-ground pool": "above ground pool",

    "heating electric": "electric heating",
    "heating: electric": "electric heating",
    "split system heating / cooling": "split system heating and cooling",
    "split system heating & cooling": "split system heating and cooling",
    "hyrdonic heating": "hydronic heating",
    "heating other": "heating: other",
    "carports": "carport",
    "electrical cooktop": "electric cooktop",
    "backyard access": "access to backyard",

    # euro laundry group (merged together — these are all the same category)
    "euro laundry": "laundry: euro",
    "laundry: european laundry": "laundry: euro",
    "laundry: european": "laundry: euro",
    "european laundry": "laundry: euro",

    "reverse cycle ac": "reverse cycle",
    "integrated fridge/freezer": "intergrated frige/freezer",  
    # NOTE: canonical form has a typo — see below
}

**Apply the merge mapping.** Converts `merge_map` into a Spark map literal, then transforms every element of `features_array` through it — replacing known variants with their canonical form, and leaving unmapped values unchanged via `coalesce`.

In [14]:
from itertools import chain

# Build a Spark map literal from the Python merge_map dict
mapping_col = F.create_map(
    [F.lit(x) for pair in merge_map.items() for x in pair])

# Apply it: for each element in features_array, look it up in the map;
# if not found (not in merge_map), keep the original value via coalesce
df_raw = df_raw.withColumn(
    "features_array",
    F.transform("features_array", lambda x: F.coalesce(mapping_col[x], x))
)

**Verify the merge worked.** Re-explodes and re-counts unique features after the merge, printing the total unique count and top 50 by frequency — confirming duplicates collapsed as intended.

In [15]:
feature_counts_after = (
    df_raw
    .select(F.explode("features_array").alias("feature"))
    .groupBy("feature")
    .count()
    .orderBy(F.desc("count"))
)
print(f"Total unique feature strings after merge: \
      {feature_counts_after.count()}")
feature_counts_after.show(50, truncate=False)

Total unique feature strings after merge:       481
+-----------------------------+-----+
|feature                      |count|
+-----------------------------+-----+
|built in wardrobes           |6690 |
|heating                      |5710 |
|dishwasher                   |5456 |
|secure parking               |4966 |
|internal laundry             |3997 |
|air conditioning             |3796 |
|balcony / deck               |2510 |
|gas                          |2421 |
|floorboards                  |1904 |
|bath                         |1881 |
|intercom                     |1629 |
|ensuite                      |1621 |
|fully fenced                 |1433 |
|study                        |1280 |
|furnished                    |1128 |
|split system air conditioning|1089 |
|pets allowed                 |1047 |
|close to shops               |965  |
|split system heating         |948  |
|close to transport           |947  |
|close to schools             |932  |
|remote garage                |919  

**Handle nulls in the array column.** Replaces any null `features_array` values with an empty array (so downstream steps don't choke on nulls), then checks and prints that zero nulls remain.

In [16]:
from pyspark.sql.types import ArrayType, StringType

df_raw = df_raw.withColumn(
    "features_array",
    F.coalesce(F.col("features_array"), F.array().cast(ArrayType(StringType())))
)

# Confirm no nulls remain before fitting
null_count = df_raw.filter(F.col("features_array").isNull()).count()
print(f"Remaining nulls in features_array: {null_count}")

Remaining nulls in features_array: 0


**Fit a CountVectorizer.** Fits Spark ML's `CountVectorizer` on `features_array` in binary mode (presence/absence, not counts), keeping only features appearing in at least 100 listings (`minDF=100`). Transforms the data and prints the resulting vocabulary size and the vocabulary list itself.

In [17]:
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="features_array",
    outputCol="features_vector",
    binary=True,
    minDF=100
)
cv_model = cv.fit(df_raw)
df_raw = cv_model.transform(df_raw)

print(f"Vocabulary size after minDF filter: {len(cv_model.vocabulary)}")
print(cv_model.vocabulary)

Vocabulary size after minDF filter: 52
['built in wardrobes', 'heating', 'dishwasher', 'secure parking', 'internal laundry', 'air conditioning', 'balcony / deck', 'gas', 'floorboards', 'bath', 'intercom', 'ensuite', 'fully fenced', 'study', 'furnished', 'split system air conditioning', 'pets allowed', 'close to shops', 'split system heating', 'close to transport', 'close to schools', 'remote garage', 'shed', 'ducted heating', 'garden / courtyard', 'gym', 'outdoor entertainment area', 'balcony', 'broadband internet access', 'alarm system', 'swimming pool', 'courtyard', 'solar panels', 'fireplace(s)', 'pool/spa count: 0', 'bathtub', 'ensuite(s)', 'carpeted', 'gas appliances', 'evaporative cooling', 'openable windows', 'broadband', 'rumpus room', 'ducted cooling', 'broadband internet available', 'city views', 'gas heating', 'deck', 'laundry', 'air-conditioner', 'separate dining room', 'double glazed windows']


**One-hot encode into individual columns.** Converts the sparse vector output from the CountVectorizer into a plain array, then loops through the vocabulary to create one binary (0/1) column per feature, with cleaned-up column names (e.g. `feat_split_system_air_conditioning`). Drops the intermediate vector/array columns afterward and previews the new columns.

In [18]:
from pyspark.ml.functions import vector_to_array

# Convert the sparse vector into a plain array column
df_raw = df_raw.withColumn("features_array_ohe", 
                           vector_to_array("features_vector"))

# Create one binary column per vocabulary term, named cleanly for readability
vocab = cv_model.vocabulary  # list of 52 feature names, in vector-index order

for i, feature_name in enumerate(vocab):
    col_name = "feat_" + feature_name.lower().replace(" ", "_").replace(
        "/", "_").replace("(", "").replace(")", "").replace(
            ":", "").replace("-", "_")
    df_raw = df_raw.withColumn(col_name, 
                               F.col("features_array_ohe")[i].cast("int"))

# Drop intermediate columns you no longer need
df_raw = df_raw.drop("features_vector", "features_array_ohe")

print(f"Added {len(vocab)} one-hot feature columns.")
df_raw.select([c for c in df_raw.columns if c.startswith("feat_")]).show(5)

Added 52 one-hot feature columns.
+-----------------------+------------+---------------+-------------------+---------------------+---------------------+-------------------+--------+----------------+---------+-------------+------------+-----------------+----------+--------------+----------------------------------+-----------------+-------------------+-------------------------+-----------------------+---------------------+------------------+---------+-------------------+-----------------------+--------+-------------------------------+------------+------------------------------+-----------------+------------------+--------------+-----------------+---------------+---------------------+------------+-------------+-------------+-------------------+------------------------+---------------------+--------------+----------------+-------------------+---------------------------------+---------------+----------------+---------+------------+--------------------+-------------------------+-------------

**Drop redundant intermediate columns.** Removes `features_array` and `structured_features_clean` since they've served their purpose and are no longer needed.

In [19]:
## Removing any redundant columns that are no longer
#  needed after one-hot encoding
df_raw = df_raw.drop("features_array", "structured_features_clean")

**Final null check.** Runs the `null_check` helper (defined earlier in the notebook) on the current state of `df_raw` to confirm no unexpected nulls have crept in after all this feature engineering.

In [20]:
null_check(df_raw)

Total rows: 12,717
Rows with at least one null value:           12,717
Percentage of rows with atleast one NULL: 100.0%

listing_id                           0
suburb                               0
postcode                             0
weekly_rent                        279
bond                               782
available_date                     136
date_listed                          4
days_listed                          4
bedrooms                           125
bathrooms                           51
carspaces                        1,803
property_type                        0
address                            104
lat                                  4
lon                                  4
scraped_date                         0
domain_page_id                       1
property_id                     10,145
photo_count                          4
video_count                          4
floorplans_count                     4
virtual_tour                         4
primary_type         

 *Lets persist what we have done with `structued_features` for ease of use in further data cleaning*

**Save the curated output.** Writes the fully cleaned DataFrame to `vic_rentals_curated_1.parquet` (overwriting any existing file), then prints the final row/column count as a confirmation of what was saved.

In [21]:
# Save the cleaned DataFrame to Parquet format
output_path = "../../rea_data/curated/vic_rentals_curated_1.parquet"

df_raw.write.mode("overwrite").parquet(output_path)

print(f"Shape: {df_raw.count():,} rows, {len(df_raw.columns)} columns")

Shape: 12,717 rows, 84 columns
